# Pre-flop action prediction

Trains the feed-forward network from `feed_forward_preflop.py` to predict a player's pre-flop action (fold, call, raise), following the same steps as that script's `main()`. The saved outputs below are from a 2M row run, so the notebook doesn't need to be re-run to view the results.

## Setup

In [ ]:
# std libs
import os
import subprocess
from pathlib import Path

from feed_forward_preflop import (
    DEFAULT_SAMPLE_HANDS,
    PokerDataset,
    PreflopEvaluator,
    PreflopNet,
    PreflopTraining,
)

PROJECT_ROOT = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip())

# NOTE: assumes the coursera user is always `coder`, may need to adjust if needed
if os.getenv("USER") == "coder":
    %pip install -r "{PROJECT_ROOT}/requirements_coursera.txt"
else:
    %pip install -r "{PROJECT_ROOT}/requirements.txt"

# external libs
import torch

DATA_PATH = PROJECT_ROOT / "parquet" / "decisions"
MAX_ROWS = None
SECTION_ROWS = 5000000
TRAIN_BATCH_SIZE = 4000
EPOCHS = 3
THREADS = 2
SEED = 0
PLAYER_DIM = 16

torch.manual_seed(SEED)
torch.set_num_threads(THREADS)


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Load the dataset

In [ ]:
dataset = PokerDataset(DATA_PATH, MAX_ROWS, SECTION_ROWS, SEED, enable_players = PLAYER_DIM > 0)

# training players: 247,354


## Build and train the model

In [ ]:
model = PreflopNet(dataset.n_players, player_dim = PLAYER_DIM)

PreflopTraining(model, dataset, TRAIN_BATCH_SIZE).run_training(EPOCHS)

## Test accuracy

In [ ]:
evaluator = PreflopEvaluator(model, dataset)
evaluator.evaluate_model().print_accuracy()


test results          rows  accuracy
combined           200,449     0.779
cards_known         11,216     0.803
cards_unknown      189,233     0.778


## Sample hands

Predicted action probabilities for the first player to act in a 6 player game, with no raises and 100 big blind stacks.

In [ ]:
evaluator.print_example_hands(DEFAULT_SAMPLE_HANDS)


Select hands sample
	hand     fold    call   raise
	AA       0.00    0.17    0.83
	AKs      0.00    0.15    0.85
	JTs      0.00    0.30    0.70
	72o      0.02    0.84    0.14
	??       0.80    0.06    0.15
